### Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [ ]:
dataset = pd.read_parquet("../data/processed/cleaned_dataset.parquet")
df = pd.DataFrame(dataset)
df.head()

In [ ]:
df['abstract'][0]

In [ ]:
print(df.shape)

In [ ]:
# Add this after Cell 5 (the cleaning cell)
# Your parquet already has cleaned data so this is just 
# creating the analysis window

df_analysis = df[df['year'].between(2006, 2022)].copy()
df_analysis = df_analysis.reset_index(drop=True)

print(f"Analysis window: {len(df_analysis)} papers")
print(df_analysis['year'].value_counts().sort_index())

### TF-IDF + PCA Representation

In [ ]:
df = df.dropna(subset=["abstract"])

df["abstract"] = df["abstract"].astype(str)

# Optional: remove very short abstracts
df = df[df["abstract"].str.split().str.len() > 20]

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,      # keep manageable
    stop_words="english",
    ngram_range=(1, 2)      # unigrams + bigrams
)

X_tfidf = tfidf.fit_transform(df["abstract"])

print("TF-IDF shape:", X_tfidf.shape)

In [ ]:
tfidf

In [ ]:
feature_names = tfidf.get_feature_names_out()
print(feature_names[:20])

In [ ]:
# First check how many components explain variance
pca_full = PCA(n_components=50)
tfidf_dense = X_tfidf.toarray()
pca_full.fit(tfidf_dense)

# Plot explained variance
plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o', markersize=3)
plt.axhline(y=0.5, color='red', linestyle='--', label='50% variance')
plt.axhline(y=0.8, color='orange', linestyle='--', label='80% variance')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('PCA Explained Variance — TF-IDF')
plt.legend()
plt.grid(True, alpha=0.3)
# plt.savefig("outputs/figures/pca_tfidf_explained_variance.png", dpi=150)
plt.show()


cumvar = np.cumsum(pca_full.explained_variance_ratio_)

def components_for_threshold(cumvar, threshold):
    indices = np.where(cumvar >= threshold)[0]
    if len(indices) == 0:
        return f"Not reached within {len(cumvar)} components (max={cumvar[-1]:.3f})"
    return indices[0] + 1

print(f"Components for 50% variance: {components_for_threshold(cumvar, 0.5)}")
print(f"Components for 80% variance: {components_for_threshold(cumvar, 0.8)}")
print(f"Variance captured by 50 components: {cumvar[-1]:.3f}")


In [ ]:
# Check what words load most strongly on PC1
feature_names = tfidf.get_feature_names_out()
pc1_loadings = pca_full.components_[0]
pc2_loadings = pca_full.components_[1]

# Top words driving PC1
top_pc1 = pd.Series(pc1_loadings, index=feature_names).nlargest(15)
top_pc2 = pd.Series(pc2_loadings, index=feature_names).nlargest(15)

print("Top words on PC1:")
print(top_pc1)
print("\nTop words on PC2:")
print(top_pc2)

In [ ]:
from sklearn.preprocessing import normalize

tfidf_normalised = normalize(X_tfidf, norm='l2')
tfidf_dense_norm = tfidf_normalised.toarray()

pca_norm = PCA(n_components=50, random_state=42)
pca_norm.fit(tfidf_dense_norm)

plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(pca_norm.explained_variance_ratio_), marker='o', markersize=3)
plt.title('PCA Explained Variance — L2 Normalised TF-IDF')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.grid(True, alpha=0.3)
plt.show()

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

def components_for_threshold(cumvar, threshold):
    indices = np.where(cumvar >= threshold)[0]
    if len(indices) == 0:
        return f"Not reached within {len(cumvar)} components (max={cumvar[-1]:.3f})"
    return indices[0] + 1

print(f"Components for 50% variance: {components_for_threshold(cumvar, 0.5)}")
print(f"Components for 80% variance: {components_for_threshold(cumvar, 0.8)}")
print(f"Variance captured by 50 components: {cumvar[-1]:.3f}")


## Insight on PCA

Applying PCA directly to TF-IDF representations highlights a key limitation of linear dimensionality reduction for high-dimensional sparse text data. Across 50 components, the cumulative explained variance reaches only 14%, with no clear elbow in the spectrum, indicating that variance is distributed across many weakly correlated dimensions rather than concentrated in a few dominant directions. This behavior is typical of bag-of-words representations, where documents are described by large vocabularies with limited global structure.

Inspection of the first principal component shows that it primarily captures differences in subfield vocabulary (e.g., cs.CL vs cs.LG), rather than meaningful temporal variation. In this dataset, PCA therefore reflects shifts in category composition more strongly than genuine topic evolution over time. As a result, low-dimensional PCA projections (e.g., 2D visualizations) should be interpreted with caution, as they provide a highly compressed and potentially misleading view of a much richer high-dimensional space.

#### PC1 and PC2 centroid movement over time plots

In [ ]:
from sklearn.decomposition import PCA

# Refit on df_analysis
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english", 
    ngram_range=(1, 2)
)

X_tfidf_analysis = tfidf.fit_transform(df_analysis['abstract'])
tfidf_dense = X_tfidf_analysis.toarray()

pca_2d = PCA(n_components=2, random_state=42)
coords = pca_2d.fit_transform(tfidf_dense)

df_analysis = df_analysis.copy()
df_analysis['pca_tfidf_x'] = coords[:, 0]
df_analysis['pca_tfidf_y'] = coords[:, 1]

print(f"Variance explained: PC1={pca_2d.explained_variance_ratio_[0]:.3f}, PC2={pca_2d.explained_variance_ratio_[1]:.3f}")
print(f"Columns now: {df_analysis.columns.tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 3))

years = sorted(df_analysis['year'].unique())
cmap = cm.get_cmap('plasma', len(years))
year_to_color = {year: cmap(i) for i, year in enumerate(years)}

# Plot 1 — individual abstracts coloured by year
ax = axes[0]
for year in years:
    mask = df_analysis['year'] == year
    ax.scatter(
        df_analysis.loc[mask, 'pca_tfidf_x'],
        df_analysis.loc[mask, 'pca_tfidf_y'],
        c=[year_to_color[year]],
        alpha=0.3, s=8, label=str(year)
    )
ax.set_title('TF-IDF PCA — Individual Abstracts', fontsize=13)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')

# Plot 2 — yearly centroids with trajectory line
ax = axes[1]
centroids_tfidf = df_analysis.groupby('year')[['pca_tfidf_x', 'pca_tfidf_y']].mean()

ax.plot(centroids_tfidf['pca_tfidf_x'], centroids_tfidf['pca_tfidf_y'],
        color='lightgray', linewidth=1.5, zorder=1)

for year in years:
    x, y = centroids_tfidf.loc[year]
    color = year_to_color[year]
    ax.scatter(x, y, c=[color], s=120, zorder=2)
    ax.annotate(str(year), (x, y), textcoords="offset points",
                xytext=(6, 4), fontsize=8)

# # Annotate known paradigm shifts
# paradigm_shifts = {2012: 'Deep Learning\n(AlexNet)', 2017: 'Transformer', 2018: 'BERT'}
# for year, label in paradigm_shifts.items():
#     if year in centroids_tfidf.index:
#         x, y = centroids_tfidf.loc[year]
#         ax.annotate(label,
#                     xy=(x, y),
#                     xytext=(x + 0.05, y + 0.05),
#                     fontsize=8, color='red',
#                     arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

# Annotate known paradigm shifts
paradigm_shifts = {
    2012: ('Deep Learning\n(AlexNet)', (-0.07, 0.025)),
    2017: ('Transformer', (-0.01, 0.008)),
    2018: ('BERT', (0.025, 0.004))
}

for year, (label, (x, y)) in paradigm_shifts.items():
    if year in centroids_tfidf.index:
        cx, cy = centroids_tfidf.loc[year]
        ax.annotate(label,
                    xy=(cx, cy),
                    xytext=(cx + 0.01, cy + 0.005),
                    fontsize=8, color='red',
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

ax.set_title('TF-IDF PCA — Yearly Centroids Trajectory', fontsize=13)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

plt.suptitle('Temporal Drift in TF-IDF Space (2010–2020)', fontsize=14, y=1.01)
plt.tight_layout()
# plt.savefig("outputs/figures/pca_tfidf_trajectory.png", dpi=150, bbox_inches='tight')
plt.show()

The trajectory shows four distinct phases:

**Phase 1 — 2007 to 2013 (upper left cluster, negative PC1, high PC2)** <br>
Early years cluster tightly together in the upper left, indicating the field had a relatively homogeneous vocabulary during this period. The high PC2 values confirm machine translation vocabulary dominated. The tight clustering suggests this was a stable paradigm period — statistical NLP and ML methods were well established and papers were relatively similar in vocabulary and framing.

**Phase 2 — 2013 to 2016 (downward movement along PC2)** <br>
The trajectory drops sharply downward from 2013 to 2016, moving from high PC2 toward zero. This corresponds exactly to the displacement of statistical MT by neural methods. The field is losing its MT-specific vocabulary signal as neural approaches absorb and replace it. Notably the horizontal PC1 position barely changes during this phase — the field is not yet moving toward the NLP/language axis, it is simply losing its old MT character.

**Phase 3 — 2016 to 2019 (rightward movement along PC1)** <br>
From 2016 the trajectory turns sharply rightward, crossing zero on PC1 and moving into positive territory. This is the transformer and BERT era — NLP vocabulary floods the corpus as language model pretraining becomes the dominant paradigm. The 2017 and 2018 points sit close together before 2019 jumps further right, suggesting BERT's impact accelerated the vocabulary shift rather than causing it suddenly.

**Phase 4 — 2019 to 2021 (downward into negative PC2)** <br>
The trajectory drops steeply downward again after 2019. This likely reflects the rise of RL-based approaches and large language model papers using a new vocabulary that is neither MT-specific nor purely linguistic — terms like reward, agent, prompt, generation pulling PC2 negative.

In [ ]:
# Calculate year-on-year centroid movement
centroid_distances = []
for i in range(1, len(years)):
    y1, y2 = years[i-1], years[i]
    c1 = centroids_tfidf.loc[y1].values
    c2 = centroids_tfidf.loc[y2].values
    dist = np.linalg.norm(c2 - c1)
    centroid_distances.append({'year_transition': f"{y1}-{y2}", 'year': y2, 'distance': dist})

dist_df = pd.DataFrame(centroid_distances)

plt.figure(figsize=(12, 5))
bars = plt.bar(dist_df['year'], dist_df['distance'], color='steelblue', alpha=0.8)

# Highlight paradigm shift years
shift_years = [2012, 2017, 2018]
for i, row in dist_df.iterrows():
    if row['year'] in shift_years:
        bars[i].set_color('crimson')
        bars[i].set_alpha(1.0)

plt.xlabel('Year')
plt.ylabel('Euclidean distance between centroids')
plt.title('Year-on-Year Centroid Distance — TF-IDF Space\n(Red bars = known paradigm shift years)')
plt.xticks(dist_df['year'], rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
# plt.savefig("outputs/figures/centroid_distance_tfidf.png", dpi=150)
plt.show()

print("\nLargest year-on-year shifts:")
print(dist_df.nlargest(3, 'distance')[['year_transition', 'distance']])

In [ ]:
# Check centroid movement along each PC separately
centroids = df_analysis.groupby('year')[['pca_tfidf_x', 'pca_tfidf_y']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# PC1 over time — NLP axis
axes[0].plot(centroids.index, centroids['pca_tfidf_x'], marker='o', color='steelblue')
axes[0].set_title('PC1 (NLP/language axis) over time')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Mean PC1 score')
axes[0].grid(True, alpha=0.3)

# PC2 over time — MT vs RL axis  
axes[1].plot(centroids.index, centroids['pca_tfidf_y'], marker='o', color='crimson')
axes[1].set_title('PC2 (MT vs RL axis) over time')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Mean PC2 score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig("outputs/figures/pc1_pc2_over_time.png", dpi=150)
plt.show()

### Analysis
- PC1 increases over time — the corpus is becoming more NLP-vocabulary-heavy, consistent with Thread 1's theory→application finding since NLP is the more applied subfield

### TO-DO: Overall assessment of TF-IDF PCA

In [ ]:
# Create cat_combo column from categories
def get_relevant_cats(cat_string):
    relevant = []
    for cat in ['cs.lg', 'cs.cl', 'cs.ai']:
        if cat in str(cat_string):
            relevant.append(cat)
    return '+'.join(sorted(relevant)) if relevant else 'other'

df_analysis = df_analysis.copy()
df_analysis['cat_combo'] = df_analysis['category'].apply(get_relevant_cats)

# Verify it worked
print(df_analysis['cat_combo'].value_counts())

In [ ]:
# Does category composition change over time?
cat_by_year = df_analysis.groupby(['year', 'cat_combo']).size().unstack(fill_value=0)
cat_by_year_pct = cat_by_year.div(cat_by_year.sum(axis=1), axis=0)

cat_by_year_pct.plot(figsize=(12, 5), marker='o')
plt.title('Category composition over time')
plt.ylabel('Proportion of papers')
plt.xlabel('Year')
plt.legend(title='Category')
plt.grid(True, alpha=0.3)
# plt.savefig("outputs/figures/category_composition_over_time.png", dpi=150)
plt.show()

_PC1 separates NLP-vocabulary-heavy papers (cs.CL) from algorithm/systems papers (cs.LG, cs.AI). Temporal drift along PC1 therefore reflects both genuine topic evolution and shifts in category composition within the corpus. These two effects cannot be fully disentangled in this representation, which is a limitation of the TF-IDF approach on a multi-category corpus — and one motivation for comparing against sentence embeddings, which capture meaning more independently of surface vocabulary._

In [ ]:
from sklearn.decomposition import TruncatedSVD

results = []

for cat in ['cs.cl', 'cs.lg', 'cs.ai']:
    subset = df_analysis[df_analysis['cat_combo'] == cat].copy()
    
    if len(subset) < 50:
        continue
    
    # Fit TF-IDF on this category only
    tfidf_cat = TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2)
    )
    X = tfidf_cat.fit_transform(subset['abstract'])
    
    # TruncatedSVD on this category only
    svd = TruncatedSVD(n_components=2, random_state=42)
    coords = svd.fit_transform(X)
    
    subset['svd_x'] = coords[:, 0]
    subset['svd_y'] = coords[:, 1]
    subset['category'] = cat
    results.append(subset)

df_by_cat = pd.concat(results).reset_index(drop=True)

# Now plot centroids per category — temporal drift within each
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, cat in zip(axes, ['cs.cl', 'cs.lg', 'cs.ai']):
    subset = df_by_cat[df_by_cat['category'] == cat]
    centroids = subset.groupby('year')[['svd_x', 'svd_y']].mean()
    
    years = centroids.index.tolist()
    cmap = cm.get_cmap('plasma', len(years))
    
    ax.plot(centroids['svd_x'], centroids['svd_y'],
            color='lightgray', linewidth=1.5, zorder=1)
    
    for i, year in enumerate(years):
        x, y = centroids.loc[year]
        ax.scatter(x, y, c=[cmap(i)], s=100, zorder=2)
        ax.annotate(str(year), (x, y),
                    textcoords="offset points",
                    xytext=(5, 3), fontsize=7)
    
    ax.set_title(f'{cat} — Temporal Drift')
    ax.set_xlabel('SVD1')
    ax.set_ylabel('SVD2')
    ax.grid(True, alpha=0.3)

plt.suptitle('Temporal Drift Within Each Category', fontsize=13)
plt.tight_layout()
# plt.savefig("../outputs/figures/temporal_drift_by_category.png", dpi=150)
plt.show()

In [ ]:
# Check what SVD1 and SVD2 capture
feature_names = tfidf.get_feature_names_out()

top_svd1 = pd.Series(svd.components_[0], index=feature_names).nlargest(15)
top_svd2 = pd.Series(svd.components_[1], index=feature_names).nlargest(15)

print("Top words on SVD1:")
print(top_svd1)
print("\nTop words on SVD2:")
print(top_svd2)

## Build Sentence-BERT embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
texts = df["abstract"].tolist()

X_embed = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

print("Embedding shape:", X_embed.shape)